In [1]:
import requests
from bs4 import BeautifulSoup
import json
import time
from urllib.parse import urljoin

In [2]:
BASE_URL = "https://library.utmn.ru"
SEARCH_URL = urljoin(BASE_URL, "/search/result")

In [3]:
# f=documentType:DT03 - фильтр по типу документа "ВКР"
params = {
    "f": "documentType:DT03"
}

In [4]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

In [5]:
def get_soup(url, params=None):
    """Получает HTML-страницу и возвращает объект BeautifulSoup."""
    try:
        response = requests.get(url, params=params, headers=HEADERS)
        response.raise_for_status()
        # Явно указываем кодировку, так как сайт может отдавать в UTF-8
        response.encoding = 'utf-8'
        return BeautifulSoup(response.text, 'html.parser')
    except requests.exceptions.RequestException as e:
        print(f"Ошибка при загрузке страницы: {e}")
        return None

In [6]:
def parse_search_page(soup):
    """Парсит страницу с результатами поиска и возвращает список ссылок на детальные страницы."""
    detail_links = []
    # На странице результаты находятся в таблице с классом 'resultTable'
    # Каждая запись - это <tr> с классом 'resultRow' или подобным
    # В предоставленном коде нет примера этой таблицы, поэтому мы ищем ссылки в 'td' с классом 'title'
    # Предполагаем, что ссылка на детальную страницу находится в первом столбце с названием
    for row in soup.find_all('tr', class_='resultRow'):
        title_cell = row.find('td', class_='title')
        if title_cell:
            link = title_cell.find('a')
            if link and link.get('href'):
                # Ссылка может быть относительной, преобразуем в абсолютную
                full_link = urljoin(BASE_URL, link['href'])
                detail_links.append(full_link)

    # Альтернативный поиск, если классы отличаются
    if not detail_links:
        # Ищем все ссылки, которые ведут на /dl/.../info
        for link in soup.find_all('a', href=True):
            href = link['href']
            if '/dl/' in href and href.endswith('/info'):
                full_link = urljoin(BASE_URL, href)
                if full_link not in detail_links:
                    detail_links.append(full_link)

    return detail_links

In [8]:
def get_next_page_url(soup):
    """Ищет ссылку на следующую страницу."""
    # Обычно кнопка "Следующая" находится в <a> с текстом "Вперед" или ">"
    next_link = soup.find('a', string='Вперёд')
    if not next_link:
        next_link = soup.find('a', string='>')
    if next_link and next_link.get('href'):
        return urljoin(BASE_URL, next_link['href'])
    return None

In [7]:
def parse_detail_page(detail_url):
    """Парсит страницу с детальной информацией о работе."""
    print(f"Парсинг детальной страницы: {detail_url}")
    soup = get_soup(detail_url)
    if not soup:
        return None

    data = {
        "title": None,
        "authors": [],
        "year": None,
        "abstract": None,
        "source_url": detail_url
    }

    # --- Извлечение названия и авторов ---
    # Название и авторы находятся в теге <h1 class="v2">
    title_tag = soup.find('h1', class_='v2')
    if title_tag:
        full_title = title_tag.get_text(strip=True)
        # Пример: "Семкин, Павел Юрьевич. Синтез, свойства,... = Synthesis, ..."
        # Разделяем по точке, чтобы отделить авторов от названия
        parts = full_title.split('. ', 1)
        if len(parts) == 2:
            authors_part, title_part = parts
            # Авторы могут быть перечислены через запятую или точку с запятой
            # Ожидаемый формат: "Семкин, Павел Юрьевич" или "Иванов И.И., Петров П.П."
            # Простой способ: разбить по ", " или "; " и убрать лишнее
            potential_authors = authors_part.split(', ')
            if len(potential_authors) > 1:
                # Если есть запятая, значит, это фамилия и инициалы
                data["authors"] = [authors_part]
            else:
                # Иначе, возможно, перечислены через точку с запятой
                data["authors"] = [name.strip() for name in authors_part.split('; ') if name.strip()]
            
            data["title"] = title_part
        else:
            # Если разделение не удалось, берем все как название
            data["title"] = full_title

    # --- Извлечение года ---
    # Год можно найти в тексте описания или в URL
    # Ищем в описании: "Тюмень, 2025"
    description_div = soup.find('div', id='description')
    if description_div:
        text = description_div.get_text()
        # Ищем год в формате 20XX или 19XX
        import re
        year_match = re.search(r'(19|20)\d{2}', text)
        if year_match:
            data["year"] = int(year_match.group(0))

    # --- Извлечение аннотации ---
    # Аннотация находится в блоке <div class="accordion"> с заголовком "Аннотация"
    accordions = soup.find_all('div', class_='accordion')
    for accordion in accordions:
        header = accordion.find('h2')
        if header and 'Аннотация' in header.get_text():
            # Текст аннотации находится в следующем div
            content_div = accordion.find('div')
            if content_div:
                # Берем все абзацы (p) внутри
                abstract_paragraphs = content_div.find_all('p', class_='annotation')
                if abstract_paragraphs:
                    # Берем русскую версию (первый абзац) или объединяем все
                    data["abstract"] = '\n'.join([p.get_text(strip=True) for p in abstract_paragraphs])
                else:
                    # Если нет класса annotation, берем просто текст
                    data["abstract"] = content_div.get_text(strip=True)
            break

    # Если год не найден в описании, попробуем извлечь из URL
    if not data["year"]:
        import re
        year_match = re.search(r'/(202[4-9]|20[3-9][0-9])/', detail_url)
        if year_match:
            data["year"] = int(year_match.group(1))

    return data

In [9]:
def main():
    all_theses = []
    next_url = SEARCH_URL
    params = {"f": "documentType:DT03"}  # Фильтр для ВКР

    page_count = 0
    while next_url:
        page_count += 1
        print(f"Обработка страницы {page_count}: {next_url}")

        soup = get_soup(next_url, params=params if next_url == SEARCH_URL else None)
        if not soup:
            break

        # Получаем ссылки на детальные страницы
        detail_links = parse_search_page(soup)
        if not detail_links:
            print("Не найдено ссылок на детальные страницы. Возможно, изменилась структура сайта.")
            # Для отладки можно сохранить HTML
            # with open(f"page_{page_count}.html", "w", encoding="utf-8") as f:
            #     f.write(str(soup))
            break

        print(f"Найдено {len(detail_links)} работ на странице.")

        for link in detail_links:
            thesis_data = parse_detail_page(link)
            if thesis_data and thesis_data.get("title"):
                all_theses.append(thesis_data)
                print(f"  Добавлена работа: {thesis_data['title'][:50]}...")
            time.sleep(0.5)  # Вежливая задержка, чтобы не перегружать сервер

        # Переход на следующую страницу
        # Если параметры были в query string, они сохраняются в href
        next_url = get_next_page_url(soup)
        time.sleep(1)

    # Сохранение результатов в JSON
    with open('libtheses.json', 'w', encoding='utf-8') as f:
        json.dump(all_theses, f, ensure_ascii=False, indent=2)

    print(f"Готово! Собрано {len(all_theses)} записей. Результат сохранен в libtheses.json")

In [10]:
if __name__ == "__main__":
    main()

Обработка страницы 1: https://library.utmn.ru/search/result
Не найдено ссылок на детальные страницы. Возможно, изменилась структура сайта.
Готово! Собрано 0 записей. Результат сохранен в libtheses.json
